In [ ]:
import logfire
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

logfire.configure()

Logfire project URL: 
https://logfire-us.pydantic.dev/amaragrawal48/ats-gap-analyser


In [4]:
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_READ_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [12]:
import pandas as pd

result = logfire_query_client.query_json_rows(sql="""
    SELECT 
        trace_id,
        created_at,
        attributes->>'total_tokens' as total_tokens,
        attributes->>'prompt_tokens' as prompt_tokens,
        attributes->>'completion_tokens' as completion_tokens
    FROM records
    WHERE message = 'token usage'
    ORDER BY created_at DESC
    LIMIT 50
""")

In [13]:
df = pd.DataFrame(result['rows'])
df['total_tokens'] = pd.to_numeric(df['total_tokens'])
df['prompt_tokens'] = pd.to_numeric(df['prompt_tokens'])
df['completion_tokens'] = pd.to_numeric(df['completion_tokens'])
df['created_at'] = pd.to_datetime(df['created_at'])

print(df)
print(f"\nTotal tokens across all sessions: {df['total_tokens'].sum()}")
print(f"Average tokens per LLM call: {df['total_tokens'].mean():.0f}")
print(f"Max tokens in a single call: {df['total_tokens'].max()}")

                            trace_id                       created_at  \
0   019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:14:11.703533+00:00   
1   019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:13:56.323020+00:00   
2   019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:13:41.168994+00:00   
3   019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:13:28.087198+00:00   
4   019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:13:12.437817+00:00   
5   019e08934049ce78cea3ef8737bfc37b 2026-05-08 17:12:59.270951+00:00   
6   019e08934049ce78cea3ef8737bfc37b 2026-05-08 17:12:39.550394+00:00   
7   019e08934049ce78cea3ef8737bfc37b 2026-05-08 17:12:12.845432+00:00   
8   019e0892961e82e15339f73a3c1d532b 2026-05-08 17:12:04.722693+00:00   
9   019e0892961e82e15339f73a3c1d532b 2026-05-08 17:11:33.420772+00:00   
10  019e0892056f986971b8471e5f9c4544 2026-05-08 17:11:21.284314+00:00   
11  019e0892056f986971b8471e5f9c4544 2026-05-08 17:11:05.652875+00:00   
12  019e0892056f986971b8471e5f9c4544 2026-05-08 17:

In [15]:
result_scores = logfire_query_client.query_json_rows(sql="""
    SELECT 
        trace_id,
        created_at,
        attributes->>'match_score' as match_score
    FROM records
    WHERE message = 'cv scored'
    ORDER BY created_at DESC
    LIMIT 50
""")

df_scores = pd.DataFrame(result_scores['rows'])
df_scores['match_score'] = pd.to_numeric(df_scores['match_score'])
df_scores['created_at'] = pd.to_datetime(df_scores['created_at'])

print(df_scores)
print(f"\nAverage match score: {df_scores['match_score'].mean():.1f}")
print(f"Min: {df_scores['match_score'].min()}, Max: {df_scores['match_score'].max()}")
print(f"Score distribution:\n{df_scores['match_score'].value_counts().sort_index()}")

                           trace_id                       created_at  \
0  019e08941447ecc5d0ce1300a1f60635 2026-05-08 17:13:28.087198+00:00   
1  019e08934049ce78cea3ef8737bfc37b 2026-05-08 17:12:27.979471+00:00   
2  019e0892961e82e15339f73a3c1d532b 2026-05-08 17:11:43.525892+00:00   
3  019e0892056f986971b8471e5f9c4544 2026-05-08 17:10:52.448219+00:00   
4  019e0891daf841664a85137b087b7906 2026-05-08 17:10:40.617368+00:00   
5  019e08526b5a67a5d5d95745304fac23 2026-05-08 16:01:23.144908+00:00   

   match_score  
0           65  
1           70  
2           75  
3           30  
4           90  
5           75  

Average match score: 67.5
Min: 30, Max: 90
Score distribution:
match_score
30    1
65    1
70    1
75    2
90    1
Name: count, dtype: int64


In [16]:
df_per_session = df.groupby('trace_id').agg(
    total_tokens=('total_tokens', 'sum'),
    llm_calls=('total_tokens', 'count')
).reset_index()

print(df_per_session)
print(f"\nAverage tokens per session: {df_per_session['total_tokens'].mean():.0f}")
print(f"Average LLM calls per session: {df_per_session['llm_calls'].mean():.1f}")

                           trace_id  total_tokens  llm_calls
0  019e08526b5a67a5d5d95745304fac23          6014          5
1  019e0891daf841664a85137b087b7906          5296          2
2  019e0892056f986971b8471e5f9c4544         10576          5
3  019e0892961e82e15339f73a3c1d532b          5377          2
4  019e08934049ce78cea3ef8737bfc37b          8683          3
5  019e08941447ecc5d0ce1300a1f60635         11784          5

Average tokens per session: 7955
Average LLM calls per session: 3.7
